# Week 3 — Preprocessing (STARTER)
### The Computer Vision Internship | VisionAI Lagos

Tunde's rule: six steps between raw PNG and model input. All six must be correct or training fails silently.
1. Load (PIL) 2. Resize 3. Augment (train only) 4. ToTensor 5. Normalize 6. Batch

## Part 1 — Augmentation Visualisation

> ⏸️ **Pause and Predict**
> If you apply `RandomRotation(15)` to a satellite patch, is the rotated image still a valid satellite image? For what rotation angles does this hold?

In [ ]:
row=df[df["land_use"]=="residential"].iloc[0]
img_orig=Image.open(IMG_DIR/row["filename"])
fig,axes=plt.subplots(2,6,figsize=(18,6))
axes[0][0].imshow(img_orig); axes[0][0].set_title("Original",fontweight="bold"); axes[0][0].axis("off")
for i in range(1,6):
    aug=train_tfm(img_orig)
    disp=aug.numpy().transpose(1,2,0)*np.array(STD)+np.array(MEAN)
    axes[0][i].imshow(np.clip(disp,0,1)); axes[0][i].set_title(f"Aug {i}"); axes[0][i].axis("off")
for i in range(6):
    val_t=eval_tfm(img_orig)
    disp=val_t.numpy().transpose(1,2,0)*np.array(STD)+np.array(MEAN)
    axes[1][i].imshow(np.clip(disp,0,1)); axes[1][i].set_title("No aug"); axes[1][i].axis("off")
axes[0][0].set_ylabel("Train (augmented)",fontweight="bold",rotation=0,labelpad=65,fontsize=9)
axes[1][0].set_ylabel("Val/Test (none)",fontweight="bold",rotation=0,labelpad=65,fontsize=9)
plt.suptitle("Augmentation: Train vs Val/Test",fontweight="bold")
plt.tight_layout(); plt.savefig("outputs/augmentation.png",dpi=150,bbox_inches="tight",facecolor="white"); plt.show()
print("Rule: augmentation on training set ONLY. Val/Test always use eval_tfm.")

## Part 2 — DataLoader Verification

In [ ]:
# predict: same batch from two next() calls in same epoch?
imgs,labels,cities=next(iter(train_loader))
print(f"Batch shape:  {imgs.shape}")
print(f"Labels:       {labels[:8].tolist()}")
print(f"Pixel range:  [{imgs.min():.2f}, {imgs.max():.2f}]  (normalised, can be negative)")
l1=next(iter(train_loader))[1][:4]
l2=next(iter(train_loader))[1][:4]
print(f"Two next() calls: {l1.tolist()} vs {l2.tolist()}")
print(f"Same? {l1.tolist()==l2.tolist()}  (shuffle=True → expect False)")

## Part 3 — Class Imbalance Check

In [ ]:
train_df=df[df["split"]=="train"]
counts=Counter(train_df["land_use"].values)
for cls,count in sorted(counts.items(),key=lambda x:-x[1]):
    print(f"  {cls:<20} {count:>4,} ({count/len(train_df):.1%})  {'|'*int(count/100)}")
ratio=max(counts.values())/min(counts.values())
print(f"Imbalance ratio: {ratio:.2f}x")
weights=compute_class_weight("balanced",classes=np.array(sorted(CLS2IDX.values())),
                               y=train_df["land_use"].map(CLS2IDX).values)
CLASS_WEIGHTS=torch.FloatTensor(weights)
print(f"Class weights: {dict(zip(CLASSES,weights.round(3)))}")

## Common Mistakes

| Mistake | Fix |
|---------|-----|
| Augmenting val/test | eval_tfm only |
| num_workers>0 on Windows | Use 0 |
| Forgetting `.to(device)` | Both imgs and labels |
| shuffle=True on val/test | shuffle=False |

## Challenge Task

> Implement MixUp augmentation and train the Week 5 CNN with it.
> Does it reduce the train-val gap? Does Kano per-city F1 improve?
> `mixed = lam * imgs + (1 - lam) * imgs[torch.randperm(n)]`

## Week 3 Complete

Next: `week4_communication_STARTER.ipynb`

*The Computer Vision Internship · VisionAI Lagos*

## ✅ By completing Week 3, you can now:

- Build a PyTorch Dataset class and DataLoader for satellite image batches
- Detect and document class imbalance in the training set
- Apply and justify an augmentation strategy appropriate for satellite imagery
- Verify normalisation constants match training set statistics

📤 **GitHub:** Push week3_preprocessing_STARTER.ipynb. Commit: "Week 3: Data pipeline complete, augmentation documented"


---

## 📚 Get the Full Book

This notebook is part of **The Computer Vision Internship** — Book 3 of the InternshipInABook™ Series.

👉 **[Get the book on Selar → [SELAR_LINK_PLACEHOLDER]]**

---
*InternshipInABook™ · © Sakinat Folorunso 2026 · [github.com/internshipinabook](https://github.com/internshipinabook/cv-internship-in-a-book)*
